# ASRN en el Open Kernel Challenge: `muse_attention_decode` en Colab

Prueba y mide el kernel de `openkernel/muse_attention_decode/submission.py` antes de gastar intentos en el juez (que
corre en una H200).

**GPU:** con L4 o H100 (*Entorno de ejecución → Cambiar tipo*) se prueba y se mide en la GPU. Con T4, A100 o sin GPU,
la prueba de correctitud corre en CPU con el intérprete de Triton (unos 5 minutos) y la medición se salta: Triton
solo convierte a FP8 E4M3 en GPUs de capacidad 8.9 o más (Ada, Hopper). El paso 3, `okc check`, funciona con
cualquiera: corre en la H200 del juez.

In [ ]:
import torch
cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
EN_GPU = cap >= (8, 9)
print(torch.__version__, torch.cuda.get_device_name() if cap > (0, 0) else 'sin GPU', cap,
      '| pruebas en GPU' if EN_GPU else '| pruebas en CPU con el interprete de Triton; sin medicion')

In [ ]:
!git clone -q -b claude/open-kernel-challenge-asrn-txzazw https://github.com/KIROKOWA/QWEN-ASRN /content/QWEN-ASRN
%cd /content/QWEN-ASRN/openkernel/muse_attention_decode

Los archivos del problema (`reference.py`, `checks.py`, `TASK.md`...) se bajan de la API pública del reto, sin
cuenta: lo mismo que escribe `okc init muse_attention_decode`.

In [ ]:
import json, os, urllib.request
d = json.load(urllib.request.urlopen('https://openkernelchallenge.dev/api/v1/problems/muse_attention_decode'))
os.makedirs('/content/okc_att', exist_ok=True)
for nombre, texto in d['files'].items():
    open(f'/content/okc_att/{nombre}', 'w').write(texto)
open('/content/okc_att/TASK.md', 'w').write(d['task_md'])
json.dump(d['benchmarks'], open('/content/okc_att/formas.json', 'w'))
print(sorted(os.listdir('/content/okc_att')), '| SGLang:', d['sglang_us'], 'us por', d['sglang_per'])

## 1. ¿Es correcto?
Los casos chicos y, con `--grandes`, tres del tamaño de las pruebas del juez (hasta 1M de contexto).

In [ ]:
if EN_GPU:
    !python probar_local.py /content/okc_att --grandes
else:
    !CUDA_VISIBLE_DEVICES= TRITON_INTERPRET=1 python probar_local.py /content/okc_att

## 2. ¿Qué tan rápido?
Tiempos de CUDA graph con el L2 frío, como mide el juez, en las 12 formas puntuadas. Una L4 tiene ~300 GB/s contra
4.8 TB/s de la H200, así que los números absolutos no se comparan con el tablero: sirven para comparar versiones
entre sí.

In [ ]:
if EN_GPU:
    !python medir.py /content/okc_att --repeticiones 30
else:
    print('Se salta: esta GPU no tiene FP8 en Triton. La velocidad la da okc check (paso 3).')

## 3. Mandarlo al juez (H200)
El reto solo publica `okc` para macOS y Linux, así que aquí se usa `okc_win.py`, un cliente pequeño de su API HTTP
que también sirve en Windows (`python okc_win.py login`).

La primera vez, corre la celda de abajo **tú mismo**: te da un código para GitHub, te muestra las reglas y te pide
escribir SI para aceptarlas. Para no repetirlo cada vez que Colab se reinicia, copia el token que imprime
`okc_win.py token` a los *Secretos* de Colab (la llave a la izquierda) con el nombre `OKC_TOKEN`.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OKC_TOKEN'] = userdata.get('OKC_TOKEN')
    print('token tomado de los Secretos de Colab')
except Exception:
    %run ../okc_win.py login

`check` es la misma corrida que cuenta, pero fuera del tablero (tiene su propio límite diario). Cuando esté bien, `submit`.

In [ ]:
!python ../okc_win.py check submission.py

In [ ]:
# !python ../okc_win.py submit submission.py
# !python ../okc_win.py leaderboard muse_attention_decode